# Evaluate UNet Segmentation with Skeleton Metrics

Runs `segmentation_skeleton_metrics.evaluate.evaluate(...)` on the same brain/segmentation pair used by `load_skeletons.ipynb`. Outputs a CSV report and a text summary in `metrics_out/`.

**Note:** This re-reads the SWCs from GCS (~15 min) because the metrics package builds its own labeled-graph representation that differs from `BrainDataset`. The `dataset_cache_*.pkl` does not help here.

### Imports

In [1]:
import os
import sys

from segmentation_skeleton_metrics.evaluate import evaluate
from segmentation_skeleton_metrics.utils.img_util import TensorStoreImage

# Shared dataset-config helper (segmentation-id lookup).
sys.path.insert(0, "../scripts")
from dataset_config import get_segmentation_id

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../configs/zihan_gcs_token.json"
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"

### Paths

In [2]:
# Match load_skeletons.ipynb so we evaluate the same UNet checkpoint:
# segmentation id is looked up from configs/segmentation_datasets.rtf by brain_id.
brain_id = "789202"
segmentation_id = get_segmentation_id(brain_id)
print(f"brain_id={brain_id} -> segmentation_id={segmentation_id}")

gt_path = f"gs://allen-nd-goog/ground_truth_tracings/{brain_id}/voxel"
fragments_path = f"gs://allen-nd-goog/from_google/{brain_id}/whole_brain/{segmentation_id}/swcs"
segmentation_path = f"gs://allen-nd-goog/from_google/{brain_id}/whole_brain/{segmentation_id}/"

output_dir = f"../metrics_out/{brain_id}/{segmentation_id}"
os.makedirs(output_dir, exist_ok=True)

brain_id=789202 -> segmentation_id=raw.unet_r3_ckpt_425250_seedfix_overlapfix_rempm


### Run Evaluation

Three-step pipeline run by `evaluate(...)`:
1. **Label graphs** -- read GT SWCs and label each node with the UNet segment ID it falls inside (uses `segmentation`).
2. **Detect errors** -- per edge, classify as correct / split / omit / merge by comparing endpoint labels.
3. **Compute metrics** -- # Splits, # Merges, % Split/Omit/Merged Edges, ERL, Normalized ERL, Edge Accuracy, Split Rate, Merge Rate.

Saved artifacts: `results.csv` (per-GT-SWC), `results_overview.txt` (averaged + totals).

In [3]:
# anisotropy: same (x, y, z) microns/voxel as load_skeletons.ipynb. Not
#   applied to the GT SWCs because gt_path ends in /voxel (already in voxel
#   coords); use_anisotropy=False reflects that.
anisotropy = (0.748, 0.748, 1.0)

# swap_axes=True: SWC voxel coords are stored (x, y, z); after the
# `from_google` transpose the segmentation volume's last 3 dims are
# (z, y, x). Setting swap_axes adds an additional transpose so the image's
# axes line up with the SWC's (x, y, z) ordering. Without this you get
# IndexError: OUT_OF_RANGE on the first GT skeleton labeling pass.
segmentation = TensorStoreImage(segmentation_path, swap_axes=True)

evaluate(
    gt_path,
    segmentation,
    output_dir,
    anisotropy=anisotropy,
    fragments_path=fragments_path,
    use_anisotropy=False,
    save_merges=True,
    save_fragments=False,
    verbose=True,
)

I0609 10:19:56.122885 1441899 google_auth_provider.cc:149] Using credentials at ../configs/zihan_gcs_token.json
I0609 10:19:56.124054 1441899 google_auth_provider.cc:165] Using ServiceAccount AuthProvider



(1) Load Ground Truth


Label Graphs: 100%|██████████| 12/12 [03:55<00:00, 19.63s/it]



(2) Load Fragments


Build Graphs: 100%|██████████| 63349/63349 [02:56<00:00, 358.85it/s]



(3) Compute Metrics


Added Cable Length (μm): 100%|██████████| 99/99 [09:10<00:00,  5.56s/it]



Average Results...
  # Splits: 861.2182
  # Merges: 9.7243
  % Split Edges: 0.3108
  % Omit Edges: 2.5134
  % Merged Edges: 17.6169
  ERL: 4179.2613
  Normalized ERL: 0.0056
  Edge Accuracy: 79.5592
  Split Rate: 1089.2885
  Merge Rate: 140826.3246

Total Results...
  # Splits: 10049
  # Merges: 99


### Inspect Results

In [4]:
import pandas as pd

results = pd.read_csv(os.path.join(output_dir, "results.csv"), index_col=0)
print(f"Per-SWC results ({len(results)} ground-truth skeletons):")
results

Per-SWC results (12 ground-truth skeletons):


,SWC Run Length,# Splits,# Merges,% Split Edges,% Omit Edges,% Merged Edges,ERL,Normalized ERL,Edge Accuracy,Split Rate,Merge Rate
N005-789202-SP,6.600023e+05,1061.0,9,0.421502,3.717807,8.648018,2916.69,0.0044,87.21,596.31,70298.08
N006-789202-JG,1.093291e+06,1200.0,9,0.285244,1.028602,14.679720,5038.70,0.0046,84.01,899.11,119880.77
N007-789202-PP,4.018429e+05,582.0,5,0.370824,5.146388,2.313584,2336.16,0.0058,92.17,652.36,75934.48
N010-789202-JT,4.555105e+05,1390.0,3,0.936992,4.598317,3.984077,1403.83,0.0031,90.48,309.57,143432.21
N011-789202-IG,3.589280e+05,675.0,1,0.481845,3.325998,1.795562,2027.03,0.0056,94.40,511.50,345260.61
N013-789202-KS,9.324719e+05,782.0,36,0.212215,1.740920,41.040640,3161.85,0.0034,57.01,1169.13,25396.10
N016-789202-HP,1.050328e+06,641.0,12,0.177426,0.554185,14.954279,6306.73,0.0060,84.31,1626.59,86886.94
N018-789202-JM,1.237078e+06,564.0,4,0.115830,0.217505,42.930046,4159.44,0.0034,56.74,2186.09,308238.62
N020-789202-PP,3.490545e+05,357.0,2,0.290955,0.719466,7.863124,9296.58,0.0266,91.13,967.86,172763.77
N021-789202-HP,6.863528e+05,616.0,8,0.231252,1.888460,3.209231,6072.15,0.0088,94.67,1090.59,83975.51


In [5]:
with open(os.path.join(output_dir, "results_overview.txt")) as f:
    print(f.read())


Average Results...
  # Splits: 861.2182
  # Merges: 9.7243
  % Split Edges: 0.3108
  % Omit Edges: 2.5134
  % Merged Edges: 17.6169
  ERL: 4179.2613
  Normalized ERL: 0.0056
  Edge Accuracy: 79.5592
  Split Rate: 1089.2885
  Merge Rate: 140826.3246

Total Results...
  # Splits: 10049
  # Merges: 99

